# Predict the cell cycle phase in new cells from the frozen NTC reference

This notebook scores cells that were **not** part of the NTC reference, for example the gene-knockout cells in `OPS0077_KO3_CCNB1_MCM3_CENPC_Multimodal_4i_Phase_DNA.h5ad`. It runs `scripts/ops0077_cc_transfer/apply_cc_transfer.py`. Nothing is estimated from the new cells.

**Two scores per cell**

| score | uses | how |
|---|---|---|
| `cc_phase_stain` | p21 / Rb / pRb nuclear intensities and DNA content | normalise with the frozen NTC per-well constants, project onto the frozen NTC cycle curve |
| `cc_phase_phase` | phase-contrast features only | the NTC-trained ridge circular regression |
| `cc_phase` | | `cc_phase_stain` where available (markers present and a reliable DNA measurement), otherwise `cc_phase_phase`; `cc_phase_source` says which |

The phase is circular, in [0, 1): 0 is the G1 start, it increases through S to G2/M, and 1 is the same point as 0. Each cell also gets an NTC-defined **sector** and a flag `out_of_ntc_support` (farther from the NTC cycle than 99% of NTC cells).

**Input requirements.** An h5ad with
- `layers['raw']`: the **raw, unscaled** 4i features, with the same columns as the NTC 4i file
- `obsm['X_phase_raw']` in the order of `uns['phase_var_names']`: raw, unscaled phase features
- `obs['well']` (`A/1/0`-style, matching the NTC wells), `obs['dna_content_log2']` and `obs['dna_reliable']`

The pre-scaled `X` and `X_phase` of the KO file are **not** used; they were scaled with another NTC file's constants.

**Prerequisite.** Run `18_build_ops0077_ntc_cc_phase_reference.ipynb` first (creates `reports/ops0077_cc_transfer/reference/`).

## 1. Requirements

In [ ]:
# %pip install anndata numpy pandas scipy scikit-learn joblib

## 2. Configuration and input checks

In [ ]:
import subprocess, sys, json
from pathlib import Path
ROOT = Path("..").resolve() if (Path("..") / "scripts").exists() else Path(".").resolve()
SCRIPTS = ROOT / "scripts/ops0077_cc_transfer"
def run(cmd):
    proc = subprocess.Popen([sys.executable, "-W", "ignore", *map(str, cmd)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, "failed, see the log above"

INPUT = ROOT / "data/altair/OPS0077_KO3_CCNB1_MCM3_CENPC_Multimodal_4i_Phase_DNA.h5ad"   # the file with the new cells
REFERENCE = ROOT / "reports/ops0077_cc_transfer/reference"
OUT = ROOT / "reports/ops0077_cc_transfer/ko3_scores"          # use another folder for another input file
DNA_COL, DNA_RELIABLE_COL = "dna_content_log2", "dna_reliable"

for n, p in {"new cells": INPUT, "NTC reference": REFERENCE / "ntc_cc_reference.json",
             "phase model": REFERENCE / "phase_ridge_model.joblib"}.items():
    print(f"{'OK     ' if p.exists() else 'MISSING'} {n}: {p.relative_to(ROOT)}")
assert INPUT.exists() and (REFERENCE / "ntc_cc_reference.json").exists(), "missing input; build the reference first (notebook 18)"

import anndata as ad
a = ad.read_h5ad(INPUT, backed="r")
need = {"layers['raw']": "raw" in a.layers, "obsm['X_phase_raw']": "X_phase_raw" in a.obsm, "uns['phase_var_names']": "phase_var_names" in a.uns,
        "obs['well']": "well" in a.obs, f"obs['{DNA_COL}']": DNA_COL in a.obs, f"obs['{DNA_RELIABLE_COL}']": DNA_RELIABLE_COL in a.obs}
print(need); assert all(need.values()), "the input is missing required fields"
print(a.n_obs, "cells | wells:", sorted(a.obs["well"].unique()))

## 3. Score the cells

In [ ]:
run([SCRIPTS / "apply_cc_transfer.py", ROOT, "--input", INPUT.relative_to(ROOT), "--reference", REFERENCE.relative_to(ROOT),
     "--out", OUT.relative_to(ROOT), "--dna-col", DNA_COL, "--dna-reliable-col", DNA_RELIABLE_COL])

## 4. Result

In [ ]:
import pandas as pd
res = pd.read_csv(OUT / "cc_phase_new_cells.csv", index_col=0)
print(json.dumps(json.load(open(OUT / "summary.json")), indent=1))
display(res.groupby("gene").agg(cells=("cc_phase", "size"), with_stain_score=("cc_phase_stain", lambda s: int(s.notna().sum())),
                                out_of_ntc_support=("out_of_ntc_support", "sum")) if "gene" in res else res.head())
res.head()

**Reading the coverage.** The stain score needs all three markers and a reliable DNA measurement; the other cells get the phase-contrast score only (`cc_phase_source`). The phase-contrast score is less precise (a mean error of about 0.1 turn on NTC cells).

## 5. Output files

| file | content |
|---|---|
| `cc_phase_new_cells.csv` | one row per cell: scores, sectors, axes, QC flags, carried metadata, and the input's own derived columns as `ref_*` (comparison only, never used as inputs) |
| `<input name>_cc_phase.h5ad` | copy of the input with the new columns in `obs` (prefixed `tr_`); the input file is untouched |
| `summary.json` | counts |

**Using the scores.** The phase is circular: compare phases with `((a - b + 0.5) % 1) - 0.5` or use `sin(2π·phase)` and `cos(2π·phase)`. Next: `20_ops0077_ko_cc_phase_evaluation.ipynb` for the visual evaluation.